# 📊 Notebook 01: EDA & Data Preprocessing

**Đồ án:** Xây dựng hệ thống E-commerce tích hợp Machine Learning

**Dataset:** Online Retail II (Kaggle/UCI)

**Mục tiêu notebook này:**
1. Tải và khám phá dataset Online Retail II
2. Làm sạch dữ liệu (xử lý null, trùng lặp, đơn hủy)
3. Phân tích khám phá dữ liệu (EDA) với trực quan hóa
4. Feature Engineering: tạo RFM + behavioral features
5. Tạo target variable `will_purchase` cho Random Forest
6. Lưu dữ liệu đã xử lý ra Google Drive

---

## 0. Cài đặt & Import thư viện

In [ ]:
# Cài đặt thư viện cần thiết
!pip install -q kagglehub openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Import thành công!')

## 1. Tải Dataset từ Kaggle

Dataset: [Online Retail II](https://www.kaggle.com/datasets/mathchi/online-retail-ii-data-set)

- **Nguồn gốc:** Dr. Daqing Chen, London South Bank University
- **Thời gian:** 01/12/2009 → 09/12/2011 (~2 năm)
- **Nội dung:** Giao dịch từ cửa hàng quà tặng online tại UK

In [ ]:
# Mount Google Drive để lưu kết quả
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Tạo thư mục project trên Drive
import os

PROJECT_DIR = '/content/drive/MyDrive/ecommerce-ml'
DATA_DIR = f'{PROJECT_DIR}/data'
MODEL_DIR = f'{PROJECT_DIR}/models'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'📁 Project directory: {PROJECT_DIR}')

In [ ]:
# Download dataset từ Kaggle
import kagglehub

path = kagglehub.dataset_download('mathchi/online-retail-ii-data-set')
print(f'📦 Dataset downloaded to: {path}')

# Liệt kê các file
for f in os.listdir(path):
    print(f'  📄 {f}')

In [ ]:
# Đọc cả 2 sheet của file Excel
xlsx_file = [f for f in os.listdir(path) if f.endswith('.xlsx')][0]
xlsx_path = os.path.join(path, xlsx_file)

print(f'📖 Đang đọc file: {xlsx_file}...')

df_2009_2010 = pd.read_excel(xlsx_path, sheet_name='Year 2009-2010')
df_2010_2011 = pd.read_excel(xlsx_path, sheet_name='Year 2010-2011')

# Gộp 2 sheet thành 1 DataFrame
df_raw = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

print(f'\n✅ Đã tải thành công!')
print(f'  - Sheet 2009-2010: {df_2009_2010.shape[0]:,} dòng')
print(f'  - Sheet 2010-2011: {df_2010_2011.shape[0]:,} dòng')
print(f'  - Tổng cộng: {df_raw.shape[0]:,} dòng × {df_raw.shape[1]} cột')

# Giải phóng memory
del df_2009_2010, df_2010_2011

## 2. Khám phá dữ liệu thô (Raw Data Exploration)

In [ ]:
# Xem 5 dòng đầu tiên
df_raw.head()

In [ ]:
# Thông tin tổng quan về dataset
print('=' * 60)
print('📋 THÔNG TIN DATASET')
print('=' * 60)
df_raw.info()
print(f'\n📐 Shape: {df_raw.shape}')
print(f'💾 Memory Usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

In [ ]:
# Thống kê mô tả cho các cột số
df_raw.describe()

In [ ]:
# Phân tích giá trị thiếu (Missing Values)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'Số lượng thiếu': missing,
    'Tỷ lệ (%)': missing_pct
}).sort_values('Tỷ lệ (%)', ascending=False)

print('=' * 60)
print('🔍 PHÂN TÍCH GIÁ TRỊ THIẾU (MISSING VALUES)')
print('=' * 60)
print(missing_df)

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if pct > 0 else '#2ecc71' for pct in missing_pct]
missing_pct.plot(kind='bar', ax=ax, color=colors)
ax.set_title('Tỷ lệ giá trị thiếu theo cột', fontsize=14, fontweight='bold')
ax.set_ylabel('Tỷ lệ (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Số lượng giá trị unique cho mỗi cột
print('=' * 60)
print('🔢 SỐ LƯỢNG GIÁ TRỊ UNIQUE')
print('=' * 60)
for col in df_raw.columns:
    print(f'  {col}: {df_raw[col].nunique():,} unique values')

In [ ]:
# Phân tích đơn hủy (Cancelled Orders)
df_raw['Invoice'] = df_raw['Invoice'].astype(str)
cancelled = df_raw[df_raw['Invoice'].str.startswith('C')]
normal = df_raw[~df_raw['Invoice'].str.startswith('C')]

print('=' * 60)
print('🚫 PHÂN TÍCH ĐƠN HỦY')
print('=' * 60)
print(f'  Tổng giao dịch: {len(df_raw):,}')
print(f'  Đơn thường: {len(normal):,} ({len(normal)/len(df_raw)*100:.1f}%)')
print(f'  Đơn hủy (bắt đầu bằng C): {len(cancelled):,} ({len(cancelled)/len(df_raw)*100:.1f}%)')

## 3. Làm sạch dữ liệu (Data Cleaning)

**Các bước:**
1. Loại bỏ dòng thiếu `Customer ID` (không thể phân tích hành vi nếu không biết khách hàng)
2. Loại bỏ đơn hủy (Invoice bắt đầu bằng 'C') — nhưng giữ lại thông tin hủy để tính `cancellation_rate`
3. Loại bỏ Quantity ≤ 0 và Price ≤ 0
4. Loại bỏ StockCode không hợp lệ (POST, DOT, BANK CHARGES, ...)
5. Tạo cột `TotalPrice = Quantity × Price`

In [ ]:
# Bước 0: Lưu thông tin cancelled orders TRƯỚC KHI loại bỏ
# (Cần để tính cancellation_rate cho feature engineering)

# Tính số đơn hủy per customer
df_raw['is_cancelled'] = df_raw['Invoice'].str.startswith('C').astype(int)

cancellation_per_customer = df_raw.groupby('Customer ID').agg(
    total_orders=('Invoice', 'nunique'),
    cancelled_orders=('is_cancelled', 'sum')
).reset_index()
cancellation_per_customer['cancellation_rate'] = (
    cancellation_per_customer['cancelled_orders'] / cancellation_per_customer['total_orders']
).round(4)

print(f'✅ Đã tính cancellation_rate cho {len(cancellation_per_customer):,} khách hàng')
cancellation_per_customer.head()

In [ ]:
# Bước 1-4: Làm sạch dữ liệu
print('🧹 BẮT ĐẦU LÀM SẠCH DỮ LIỆU')
print(f'  Dòng ban đầu: {len(df_raw):,}')

# Copy để giữ nguyên raw data
df = df_raw.copy()

# 1. Loại bỏ Customer ID null
df = df.dropna(subset=['Customer ID'])
print(f'  Sau loại bỏ null Customer ID: {len(df):,}')

# 2. Loại bỏ đơn hủy
df = df[~df['Invoice'].str.startswith('C')]
print(f'  Sau loại bỏ đơn hủy: {len(df):,}')

# 3. Loại bỏ Quantity <= 0 và Price <= 0
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
print(f'  Sau loại bỏ Quantity/Price <= 0: {len(df):,}')

# 4. Loại bỏ StockCode không hợp lệ (service charges, adjustments, etc.)
invalid_stock_codes = ['POST', 'DOT', 'M', 'BANK CHARGES', 'PADS',
                        'AMAZONFEE', 'CRUK', 'C2', 'D']
df = df[~df['StockCode'].isin(invalid_stock_codes)]
print(f'  Sau loại bỏ StockCode không hợp lệ: {len(df):,}')

# 5. Tạo cột TotalPrice
df['TotalPrice'] = df['Quantity'] * df['Price']

# Ép kiểu Customer ID về int
df['Customer ID'] = df['Customer ID'].astype(int)

# Đảm bảo InvoiceDate là datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f'\n✅ Hoàn tất làm sạch!')
print(f'  Dòng còn lại: {len(df):,} ({len(df)/len(df_raw)*100:.1f}% dữ liệu gốc)')
print(f'  Số khách hàng: {df["Customer ID"].nunique():,}')
print(f'  Số sản phẩm: {df["StockCode"].nunique():,}')
print(f'  Khoảng thời gian: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')

In [ ]:
# Xem lại sau khi clean
df.head(10)

In [ ]:
# Kiểm tra lại missing values sau khi clean
print('Missing values sau khi clean:')
print(df.isnull().sum())

## 4. Phân tích khám phá dữ liệu (EDA)

Trực quan hóa các khía cạnh quan trọng của dataset để hiểu rõ patterns.

### 4.1. Phân bố doanh thu theo thời gian

In [ ]:
# Doanh thu theo tháng
monthly_revenue = df.set_index('InvoiceDate').resample('M')['TotalPrice'].sum()

fig, ax = plt.subplots(figsize=(14, 6))
monthly_revenue.plot(kind='line', ax=ax, marker='o', color='#3498db', linewidth=2)
ax.fill_between(monthly_revenue.index, monthly_revenue.values, alpha=0.15, color='#3498db')
ax.set_title('📈 Doanh thu theo tháng (Monthly Revenue)', fontsize=16, fontweight='bold')
ax.set_xlabel('Tháng')
ax.set_ylabel('Doanh thu (£)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.show()

### 4.2. Phân bố theo quốc gia

In [ ]:
# Top 10 quốc gia theo doanh thu
country_revenue = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue
colors = sns.color_palette('viridis', len(country_revenue))
country_revenue.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Top 10 Quốc gia theo Doanh thu', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Doanh thu (£)')
axes[0].invert_yaxis()

# Số lượng khách hàng
country_customers = df.groupby('Country')['Customer ID'].nunique().sort_values(ascending=False).head(10)
country_customers.plot(kind='barh', ax=axes[1], color=sns.color_palette('magma', len(country_customers)))
axes[1].set_title('Top 10 Quốc gia theo Số khách hàng', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Số khách hàng')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 4.3. Top sản phẩm bán chạy

In [ ]:
# Top 15 sản phẩm theo số lượng bán
top_products = df.groupby(['StockCode', 'Description']).agg(
    total_qty=('Quantity', 'sum'),
    total_revenue=('TotalPrice', 'sum'),
    num_orders=('Invoice', 'nunique')
).sort_values('total_qty', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(14, 7))
product_names = [f"{desc[:35]}..." if len(str(desc)) > 35 else str(desc)
                 for desc in top_products.index.get_level_values('Description')]
bars = ax.barh(product_names, top_products['total_qty'],
               color=sns.color_palette('coolwarm', len(top_products)))
ax.set_title('🏆 Top 15 Sản phẩm bán chạy nhất', fontsize=16, fontweight='bold')
ax.set_xlabel('Tổng số lượng bán')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\n📊 Chi tiết:')
top_products.reset_index().head(10)

### 4.4. Patterns mua sắm theo giờ & ngày trong tuần

In [ ]:
# Tạo features thời gian
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek  # 0=Monday, 6=Sunday
df['Month'] = df['InvoiceDate'].dt.month

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Theo giờ
hourly = df.groupby('Hour')['Invoice'].nunique()
axes[0].bar(hourly.index, hourly.values, color='#e74c3c', alpha=0.8)
axes[0].set_title('🕐 Số đơn hàng theo giờ trong ngày', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Giờ')
axes[0].set_ylabel('Số đơn hàng')
axes[0].set_xticks(range(6, 22))

# Theo ngày trong tuần
daily = df.groupby('DayOfWeek')['Invoice'].nunique()
axes[1].bar([day_names[i] for i in daily.index], daily.values,
            color='#2ecc71', alpha=0.8)
axes[1].set_title('📅 Số đơn hàng theo ngày trong tuần', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Ngày')
axes[1].set_ylabel('Số đơn hàng')

plt.tight_layout()
plt.show()

### 4.5. Phân bố giá trị đơn hàng

In [ ]:
# Phân bố giá trị mỗi đơn hàng
order_values = df.groupby('Invoice')['TotalPrice'].sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full distribution
axes[0].hist(order_values, bins=100, color='#9b59b6', alpha=0.7, edgecolor='white')
axes[0].set_title('Phân bố giá trị đơn hàng (Full)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Giá trị đơn hàng (£)')
axes[0].set_ylabel('Tần suất')

# Zoomed (loại outliers)
q99 = order_values.quantile(0.99)
axes[1].hist(order_values[order_values <= q99], bins=80,
             color='#f39c12', alpha=0.7, edgecolor='white')
axes[1].set_title(f'Phân bố giá trị đơn hàng (≤ £{q99:,.0f}, 99th percentile)',
                   fontsize=14, fontweight='bold')
axes[1].set_xlabel('Giá trị đơn hàng (£)')
axes[1].set_ylabel('Tần suất')

plt.tight_layout()
plt.show()

print(f'📊 Thống kê giá trị đơn hàng:')
print(f'  Mean: £{order_values.mean():,.2f}')
print(f'  Median: £{order_values.median():,.2f}')
print(f'  Min: £{order_values.min():,.2f}')
print(f'  Max: £{order_values.max():,.2f}')

### 4.6. Phân bố số lần mua hàng của khách

In [ ]:
# Số lần mua hàng per customer
customer_order_counts = df.groupby('Customer ID')['Invoice'].nunique()

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(customer_order_counts[customer_order_counts <= 50], bins=50,
        color='#1abc9c', alpha=0.8, edgecolor='white')
ax.set_title('Phân bố số đơn hàng per khách (≤ 50 đơn)', fontsize=14, fontweight='bold')
ax.set_xlabel('Số đơn hàng')
ax.set_ylabel('Số khách hàng')
ax.axvline(customer_order_counts.median(), color='red', linestyle='--',
            label=f'Median = {customer_order_counts.median():.0f} đơn')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'📊 Thống kê số đơn hàng per khách:')
print(f'  Khách chỉ mua 1 lần: {(customer_order_counts == 1).sum():,} ({(customer_order_counts == 1).mean()*100:.1f}%)')
print(f'  Khách mua > 1 lần: {(customer_order_counts > 1).sum():,} ({(customer_order_counts > 1).mean()*100:.1f}%)')
print(f'  Khách mua > 10 lần: {(customer_order_counts > 10).sum():,} ({(customer_order_counts > 10).mean()*100:.1f}%)')

---

## 5. Feature Engineering

Tạo features ở **cấp độ khách hàng (Customer-level)** từ dữ liệu giao dịch.

### Chiến lược tạo label `will_purchase`:

```
Timeline: 01/2010 ─────────────────────── 30/06/2011 ─── 09/12/2011
          │        TRAINING PERIOD         │  PREDICTION  │
          │   (Tính features từ đây)        │   WINDOW     │
          │                                 │  (Label Y)   │
```

- **Training period**: Dữ liệu từ đầu → 30/06/2011 → dùng để tính features
- **Prediction window**: 01/07/2011 → 09/12/2011 → dùng để tạo label
- **Label**: Khách có mua hàng trong prediction window? → 1 = Có, 0 = Không

In [ ]:
# Xác định mốc thời gian
CUTOFF_DATE = pd.Timestamp('2011-06-30')
REFERENCE_DATE = CUTOFF_DATE  # Ngày tham chiếu để tính Recency

# Chia data theo thời gian
df_train_period = df[df['InvoiceDate'] <= CUTOFF_DATE]
df_predict_period = df[df['InvoiceDate'] > CUTOFF_DATE]

print(f'📅 Mốc cắt (Cutoff Date): {CUTOFF_DATE.date()}')
print(f'  Training period: {df_train_period["InvoiceDate"].min().date()} → {df_train_period["InvoiceDate"].max().date()}')
print(f'  Prediction window: {df_predict_period["InvoiceDate"].min().date()} → {df_predict_period["InvoiceDate"].max().date()}')
print(f'\n  Training records: {len(df_train_period):,}')
print(f'  Prediction records: {len(df_predict_period):,}')
print(f'  Training customers: {df_train_period["Customer ID"].nunique():,}')
print(f'  Prediction customers: {df_predict_period["Customer ID"].nunique():,}')

In [ ]:
# ═══════════════════════════════════════════════════
# TÍNH RFM FEATURES (Recency, Frequency, Monetary)
# ═══════════════════════════════════════════════════

# Group theo Customer ID trên TRAINING PERIOD
customer_data = df_train_period.groupby('Customer ID').agg(
    # Recency: số ngày từ lần mua cuối đến ngày tham chiếu
    last_purchase_date=('InvoiceDate', 'max'),
    first_purchase_date=('InvoiceDate', 'min'),

    # Frequency: số đơn hàng riêng biệt
    frequency=('Invoice', 'nunique'),

    # Monetary: tổng chi tiêu
    monetary=('TotalPrice', 'sum'),

    # Thêm features
    total_items=('Quantity', 'sum'),
    total_unique_products=('StockCode', 'nunique'),
    avg_unit_price=('Price', 'mean'),
    country=('Country', 'first'),
).reset_index()

# Recency
customer_data['recency'] = (REFERENCE_DATE - customer_data['last_purchase_date']).dt.days

# Days since first purchase (tuổi khách hàng)
customer_data['days_since_first_purchase'] = (
    REFERENCE_DATE - customer_data['first_purchase_date']
).dt.days

print(f'✅ Đã tạo RFM features cho {len(customer_data):,} khách hàng')
customer_data[['Customer ID', 'recency', 'frequency', 'monetary']].describe()

In [ ]:
# ═══════════════════════════════════════════════════
# TÍNH BEHAVIORAL FEATURES
# ═══════════════════════════════════════════════════

# --- Average Order Value ---
customer_data['avg_order_value'] = customer_data['monetary'] / customer_data['frequency']

# --- Average Items per Order ---
customer_data['avg_items_per_order'] = customer_data['total_items'] / customer_data['frequency']

# --- Average Days Between Orders ---
order_dates_per_customer = df_train_period.groupby('Customer ID')['InvoiceDate'].apply(
    lambda x: x.dt.date.unique()
)

avg_days_between = {}
for cid, dates in order_dates_per_customer.items():
    if len(dates) > 1:
        sorted_dates = sorted(dates)
        diffs = [(sorted_dates[i+1] - sorted_dates[i]).days for i in range(len(sorted_dates)-1)]
        avg_days_between[cid] = np.mean(diffs)
    else:
        avg_days_between[cid] = 0  # Chỉ mua 1 lần

customer_data['avg_days_between_orders'] = customer_data['Customer ID'].map(avg_days_between)

# --- Weekend Shopper Ratio ---
weekend_ratio = df_train_period.groupby('Customer ID').apply(
    lambda x: (x['InvoiceDate'].dt.dayofweek >= 5).mean()
).reset_index(name='is_weekend_shopper')
customer_data = customer_data.merge(weekend_ratio, on='Customer ID', how='left')

# --- Favorite Hour ---
fav_hour = df_train_period.groupby('Customer ID')['Hour'].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else 12
).reset_index(name='favorite_hour')
customer_data = customer_data.merge(fav_hour, on='Customer ID', how='left')

# --- Cancellation Rate ---
customer_data = customer_data.merge(
    cancellation_per_customer[['Customer ID', 'cancellation_rate']],
    on='Customer ID', how='left'
)
customer_data['cancellation_rate'] = customer_data['cancellation_rate'].fillna(0)

# --- Country Encoding ---
from sklearn.preprocessing import LabelEncoder
le_country = LabelEncoder()
customer_data['country_encoded'] = le_country.fit_transform(customer_data['country'])

print(f'✅ Đã tạo behavioral features!')
print(f'Tổng features: {customer_data.shape[1]} cột')
customer_data.head()

In [ ]:
# ═══════════════════════════════════════════════════
# TẠO TARGET VARIABLE: will_purchase
# ═══════════════════════════════════════════════════

# Khách hàng nào xuất hiện trong prediction window?
customers_who_purchased = set(df_predict_period['Customer ID'].unique())

customer_data['will_purchase'] = customer_data['Customer ID'].isin(
    customers_who_purchased
).astype(int)

# Thống kê class balance
purchase_counts = customer_data['will_purchase'].value_counts()
print('=' * 60)
print('🎯 TARGET VARIABLE: will_purchase')
print('=' * 60)
print(f'  Sẽ mua lại (1): {purchase_counts.get(1, 0):,} khách ({purchase_counts.get(1, 0)/len(customer_data)*100:.1f}%)')
print(f'  Không mua (0):  {purchase_counts.get(0, 0):,} khách ({purchase_counts.get(0, 0)/len(customer_data)*100:.1f}%)')
print(f'  Tỷ lệ imbalance: 1:{purchase_counts.get(0, 0)/max(purchase_counts.get(1, 1), 1):.1f}')

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#2ecc71']
purchase_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Phân bố Target Variable: will_purchase', fontsize=14, fontweight='bold')
ax.set_xlabel('Label')
ax.set_ylabel('Số khách hàng')
ax.set_xticklabels(['Không mua lại (0)', 'Sẽ mua lại (1)'], rotation=0)
for i, v in enumerate(purchase_counts.values):
    ax.text(i, v + 30, f'{v:,}', ha='center', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Phân tích Features

In [ ]:
# Danh sách features sẽ dùng cho model
FEATURE_COLUMNS = [
    'recency',
    'frequency',
    'monetary',
    'avg_order_value',
    'avg_items_per_order',
    'total_unique_products',
    'avg_days_between_orders',
    'cancellation_rate',
    'days_since_first_purchase',
    'is_weekend_shopper',
    'favorite_hour',
    'country_encoded',
    'avg_unit_price',
]

TARGET = 'will_purchase'

print(f'📋 Features ({len(FEATURE_COLUMNS)}):')
for i, f in enumerate(FEATURE_COLUMNS, 1):
    print(f'  {i:2d}. {f}')
print(f'\n🎯 Target: {TARGET}')

In [ ]:
# Distribution of features split by target
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for idx, col in enumerate(FEATURE_COLUMNS[:12]):
    ax = axes[idx]
    for label, color, name in [(0, '#e74c3c', 'Không mua'), (1, '#2ecc71', 'Sẽ mua')]:
        data = customer_data[customer_data[TARGET] == label][col]
        # Clip outliers cho visualization
        q95 = data.quantile(0.95)
        data_clipped = data[data <= q95] if q95 > 0 else data
        ax.hist(data_clipped, bins=30, alpha=0.5, label=name, color=color, density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Phân bố Features theo Target (will_purchase)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Matrix
corr_cols = FEATURE_COLUMNS + [TARGET]
corr_matrix = customer_data[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('🔗 Ma trận tương quan (Correlation Matrix)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# In tương quan với target
print('\n📊 Tương quan với will_purchase (sắp xếp theo |corr|):')
target_corr = corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
for feat, corr in target_corr.items():
    direction = '⬆️' if corr_matrix.loc[feat, TARGET] > 0 else '⬇️'
    print(f'  {direction} {feat}: {corr_matrix.loc[feat, TARGET]:+.3f}')

## 7. Lưu dữ liệu đã xử lý

Lưu tất cả kết quả ra Google Drive để dùng trong các notebook tiếp theo.

In [ ]:
# Lưu customer features dataset
customer_features_path = f'{DATA_DIR}/customer_features.csv'
customer_data.to_csv(customer_features_path, index=False)
print(f'✅ Saved: {customer_features_path}')
print(f'   Shape: {customer_data.shape}')

# Lưu clean transaction data (cho KNN product recommendation sau này)
product_data_path = f'{DATA_DIR}/transactions_clean.csv'
df.to_csv(product_data_path, index=False)
print(f'\n✅ Saved: {product_data_path}')
print(f'   Shape: {df.shape}')

# Lưu feature columns config
import json
config = {
    'feature_columns': FEATURE_COLUMNS,
    'target': TARGET,
    'cutoff_date': str(CUTOFF_DATE.date()),
    'reference_date': str(REFERENCE_DATE.date()),
    'num_customers': len(customer_data),
    'num_features': len(FEATURE_COLUMNS),
    'country_classes': le_country.classes_.tolist()
}
config_path = f'{DATA_DIR}/config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'\n✅ Saved: {config_path}')

# Lưu LabelEncoder
import joblib
joblib.dump(le_country, f'{MODEL_DIR}/label_encoder_country.joblib')
print(f'\n✅ Saved: {MODEL_DIR}/label_encoder_country.joblib')

print('\n' + '=' * 60)
print('🎉 HOÀN TẤT NOTEBOOK 01!')
print('Tiếp tục với Notebook 02: Purchase Prediction (Random Forest)')
print('=' * 60)